# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and analyzing a Croissant-based dataset using the `mlcroissant` library, referencing all datasets, record sets, fields, and columns by their `@id` fields as per the Croissant specification and best practice.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print overview
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', None)}")
print(f"License: {getattr(metadata, 'license', None)}")
print(f"Published: {getattr(metadata, 'datePublished', None)}")

## 2. Data Overview
Review available `RecordSet`s, their field `@id`s, and column `@id`s, referencing all entities by their Croissant `@id`.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.metadata.recordSets
print("Available Record Sets (@id and name):\n-----------------------------")
for rec in record_sets:
    print(f"@id: {rec.id}")
    print(f"  name: {rec.name}")
    print(f"  description: {getattr(rec, 'description', '')}")
    if hasattr(rec, 'fields'):
        print("  Fields (by @id):")
        for field in rec.fields:
            print(f"    @id: {field.id} - name: {field.name}")
            # List columns for this field
            if hasattr(field, 'columns'):
                for col in field.columns:
                    print(f"      column @id: {col.id} - name: {col.name}")
    print()
# Example: print a few record instances for the first record set
if record_sets:
    example_record_set_id = record_sets[0].id
    print(f"\nFirst 2 records from record set {example_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        print(record)
        if i > 0: break

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis.
All `record_set` and field references use their Croissant `@id`.

In [ ]:
# Extract data from each RecordSet using their @id
dataframes = {}
record_set_ids = [rec.id for rec in dataset.metadata.recordSets]
# For demonstration, load all record sets.
for rec_id in record_set_ids:
    print(f"Loading record set {rec_id}")
    records = list(dataset.records(record_set=rec_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Columns in DataFrame for {rec_id}: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {rec_id}.")

# For the remainder, we'll select the FIRST record set for EDA
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet {record_set_id} for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, and grouping using Croissant `@id` references.
Below, you can change the `numeric_field_id` and `group_field_id` to fields listed in the overview or the record set definition above.

In [ ]:
# Choose a numeric field and group field by @id (adjust to your dataset)
numeric_field_id = None
group_field_id = None

# Try to auto-detect a numeric field from fields with number/integer/float type
target_record_set = None
for rec in dataset.metadata.recordSets:
    if rec.id == record_set_id:
        target_record_set = rec
        break
if target_record_set and hasattr(target_record_set, 'fields'):
    for f in target_record_set.fields:
        dtype = getattr(f, 'dataType', '').lower()
        if dtype in ("integer", "float", "number"): # standard Croissant types
            numeric_field_id = f.id
            break
    # Pick another field for grouping (often categorical)
    for f in target_record_set.fields:
        dtype = getattr(f, 'dataType', '').lower()
        if 'str' in dtype or dtype in ("text", "string") or dtype == 'categorical':
            group_field_id = f.id
            break

if not numeric_field_id:
    # Default fallback: use first column
    numeric_field_id = df.columns[0]
if not group_field_id:
    group_field_id = df.columns[1] if len(df.columns)>1 else df.columns[0]

print(f"Numeric field for analysis (by @id): {numeric_field_id}")
print(f"Group field for grouping (by @id): {group_field_id}")

# Remove rows with missing values for numeric analysis
df_numeric = df.copy()
df_numeric = df_numeric[pd.to_numeric(df_numeric[numeric_field_id], errors='coerce').notna()]
df_numeric[numeric_field_id] = pd.to_numeric(df_numeric[numeric_field_id], errors='coerce')

# Example EDA: filter, normalize, group
threshold = df_numeric[numeric_field_id].mean() # use mean as threshold
filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}: {len(filtered_df)} records")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by another field
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Simple histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df_numeric[numeric_field_id], bins=20, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group if applicable
if group_field_id in df_numeric.columns and df_numeric[group_field_id].nunique() < 20:
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=df_numeric, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and analyze a Croissant-based dataset using the `mlcroissant` library. We referenced all dataset elements by their Croissant `@id` fields for reproducibility and consistency.

- **Exploration:** Listed all data record sets and fields (`@id`).
- **Extraction:** Loaded record data into pandas DataFrames.
- **EDA:** Processed and explored subsets of the data with normalization and grouping.
- **Visualization:** Displayed basic distributions and group statistics.

Continue your exploration by selecting other record sets and fields (using their `@id`s from the overview above), and applying detailed analyses tailored to your research questions. For more on Croissant, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/usage/).